# FIT4112 · Lab 2 · Notebook 01 — Khai phá luật kết hợp bằng FP-Growth

**Tiếp nối Lab 1:** đã biết biểu diễn basket và Apriori. Notebook này dùng **FP-Growth** để tìm cùng loại tập mục phổ biến, rồi sinh luật kết hợp.

FP-Growth tận dụng việc nén mẫu giao dịch trong FP-tree thay vì tạo và thử nhiều ứng viên như Apriori. **Không cần tự lập trình FP-tree trong Lab 2.**

**Sản phẩm:** `data/processed/fpgrowth_rules.csv`, 5 luật được chọn và phần trả lời trong báo cáo.

## 1. Chuẩn bị và đọc basket

Basket mặc định có 40 hóa đơn và 9 sản phẩm. Nếu đã làm Lab 1 với dữ liệu khác, có thể đặt `basket_bool.csv` tại `data/input/basket_lab1.csv` rồi đổi `BASKET_FILE`.

In [ ]:
from pathlib import Path
from time import perf_counter
import pandas as pd
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules

def find_root():
    for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (p / "data" / "basket_demo.csv").exists():
            return p
    raise FileNotFoundError("Không thấy data/basket_demo.csv. Hãy mở notebook từ repo hoặc notebooks/.")

ROOT = find_root()
OUTPUT = ROOT / "data" / "processed"
OUTPUT.mkdir(exist_ok=True, parents=True)

def load_basket(file_name):
    path = ROOT / file_name
    if not path.exists():
        raise FileNotFoundError(f"Không tìm thấy {path}; hãy kiểm tra đường dẫn.")
    df = pd.read_csv(path, index_col=0)
    if df.empty or df.shape[1] < 2:
        raise ValueError("Basket cần ít nhất 1 hóa đơn và 2 sản phẩm.")
    numeric = df.apply(pd.to_numeric, errors="raise")
    if not numeric.isin([0, 1]).all().all():
        raise ValueError("Basket chỉ được chứa các giá trị 0 hoặc 1.")
    return numeric.astype(bool)

def mine_rules(itemsets, min_confidence):
    if itemsets.empty or not itemsets["itemsets"].map(len).ge(2).any():
        return pd.DataFrame(columns=["antecedents", "consequents", "support", "confidence", "lift"])
    return association_rules(itemsets, metric="confidence", min_threshold=min_confidence)

def show_rules(rules):
    cols = ["antecedents_str", "consequents_str", "support", "confidence", "lift"]
    if rules.empty:
        return pd.DataFrame(columns=cols)
    out = rules.copy()
    out["antecedents_str"] = out["antecedents"].map(lambda x: ", ".join(sorted(x)))
    out["consequents_str"] = out["consequents"].map(lambda x: ", ".join(sorted(x)))
    return out[cols].sort_values(["lift", "confidence"], ascending=[False, False]).reset_index(drop=True)

print("Project root:", ROOT)

BASKET_FILE = "data/basket_demo.csv"
MIN_SUPPORT = 0.10
MIN_CONFIDENCE = 0.30
MAX_LEN = 3
basket = load_basket(BASKET_FILE)
print(f"Basket: {basket.shape[0]} hóa đơn × {basket.shape[1]} sản phẩm")
display(basket.head().astype(int))

## 2. FP-Growth – khai phá tập mục phổ biến

**Support:** tỷ lệ hóa đơn có tất cả sản phẩm thuộc một tập mục. `MIN_SUPPORT=0.10` nghĩa là tập mục phải xuất hiện trong ít nhất 10% hóa đơn.

In [ ]:
t0 = perf_counter()
itemsets = fpgrowth(basket, min_support=MIN_SUPPORT, use_colnames=True, max_len=MAX_LEN)
elapsed = (perf_counter() - t0) * 1000
itemsets["length"] = itemsets["itemsets"].map(len)
print("Số frequent itemsets:", len(itemsets))
print(f"Thời gian chạy FP-Growth (một lần): {elapsed:.3f} ms")
display(itemsets.sort_values("support", ascending=False).head(10))

## 3. Sinh luật và đọc ba chỉ số

- **support(A → B):** tỷ lệ hóa đơn chứa A và B.
- **confidence(A → B):** tỷ lệ hóa đơn chứa B trong nhóm hóa đơn có A.
- **lift(A → B):** confidence / support(B); lift > 1 cho thấy liên hệ dương, không hàm ý quan hệ nhân quả.

In [ ]:
rules = mine_rules(itemsets, MIN_CONFIDENCE)
table = show_rules(rules)
print("Số luật đạt min_confidence:", len(table))
display(table.head(15).round(3))
if len(table) < 5:
    print("Có ít hơn 5 luật; thử giảm ngưỡng trước khi chọn luật.")

## 4. Quan sát top 10 luật theo Lift và xuất CSV

Một luật có lift lớn nhưng support rất nhỏ có thể chưa hữu ích khi áp dụng; cần xem đồng thời các chỉ số.

In [ ]:
import matplotlib.pyplot as plt
if not table.empty:
    top = table.head(10).copy()
    labels = top["antecedents_str"] + " → " + top["consequents_str"]
    ax = top.set_index(labels)["lift"].sort_values().plot.barh(figsize=(9, 5), title="Top 10 luật FP-Growth theo Lift")
    ax.set_xlabel("Lift")
    plt.tight_layout()
    plt.show()
export_path = OUTPUT / "fpgrowth_rules.csv"
table.to_csv(export_path, index=False)
print("Đã xuất:", export_path)

## 5. Tự kiểm tra và TODO

**TODO A:** Chọn 5 luật tiêu biểu, ghi tiền đề (antecedents), kết quả (consequents), support, confidence, lift. Giải thích 2 luật bằng tiếng Việt.

**TODO B:** Giải thích ngắn gọn Apriori và FP-Growth giống nhau ở **đầu ra**, nhưng khác nhau ở **cách khai phá** như thế nào.

**TODO C:** Đề xuất một combo sản phẩm hoặc cách trưng bày và nêu giới hạn của dữ liệu demo.

**Tiếp theo:** mở `02_compare_apriori_fpgrowth.ipynb`.